# Brain-to-Text: Skip-Diphone + Temporal Smoothness

**Project**: Context-Aware Neural Speech Decoding with Skip-Diphone Auxiliary Supervision and Temporal Smoothness Regularization  
**Author**: Jiayu (Jarrod) Yang — Introduction to Deep Learning, Spring 2026  
**Proposal**: [../docs/proposal.pdf](../docs/proposal.pdf)

---

## Background

This project builds on the **DCoND** framework (Li et al., arXiv:2411.10657), which improves
neural speech decoding by predicting *diphones* (z_{t-1}→z_t) instead of isolated phonemes,
then marginalizing back to recover per-frame phoneme probabilities.

Two extensions are tested:

| Extension | Hypothesis |
|-----------|------------|
| **Skip-diphone auxiliary head** (z_{t-2}→z_t) | Speech motor planning encodes context beyond the immediately preceding phoneme |
| **Temporal smoothness loss** on marginalized phoneme probs | Reduces frame-level jitter and spurious CTC insertions in noisy intracortical recordings |

## Training Objective (Eq. 1)

$$\mathcal{L}_{\text{total}} = \mathcal{L}^{\text{CTC}}_{\text{phoneme}} + \alpha\,\mathcal{L}^{\text{CTC}}_{\text{std-diphone}} + \beta\,\mathcal{L}^{\text{CTC}}_{\text{skip-diphone}} + \lambda\,\mathcal{L}_{\text{smooth}}$$

$$\mathcal{L}_{\text{smooth}} = \frac{1}{T-1}\sum_{t=2}^{T}\|p_t - p_{t-1}\|_2^2$$

## Ablation Variants

| Variant | Components |
|---------|------------|
| A | Mono CTC only (NPTL baseline) |
| B | Diphone + marginalization (DCoND) |
| C | B + smoothness loss |
| D | B + skip-diphone auxiliary head |
| E | B + skip-diphone + smoothness (full model) |

---

This notebook reports:
1. Ablation table (variants A–E): PER and WER after 3-gram LM decoding
2. Smoothness weight λ sweep (variants C and E)
3. Validation loss training curves per variant

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

## 1. Ablation Table (A–E)

In [ ]:
# Fill in after running decode.py for each variant
results = {
    "Variant": ["A", "B", "C", "D", "E"],
    "Description": [
        "Mono CTC (baseline)",
        "Diphone + marginalization (DCoND)",
        "B + smoothness",
        "B + skip-diphone aux",
        "B + skip-diphone + smoothness (full)",
    ],
    "PER (3-gram) %": [None, None, None, None, None],
    "WER (3-gram) %": [None, None, None, None, None],
}

df = pd.DataFrame(results)
df

## 2. λ Sweep (Smoothness Weight)

In [ ]:
# Fill in after running variant C / E with different lambda values
lambda_vals = [0, 1e-3, 5e-3, 1e-2]
per_C = [None, None, None, None]   # variant C PER per lambda
per_E = [None, None, None, None]   # variant E PER per lambda

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(lambda_vals, per_C, marker="o", label="C (smooth only)")
ax.plot(lambda_vals, per_E, marker="s", label="E (full model)")
ax.set_xscale("symlog", linthresh=1e-4)
ax.set_xlabel("λ (smoothness weight)")
ax.set_ylabel("PER (%)")
ax.set_title("Effect of smoothness weight λ on PER")
ax.legend()
plt.tight_layout()
plt.savefig("../experiments/lambda_sweep.pdf", bbox_inches="tight")
plt.show()

## 3. Training Curves

In [ ]:
# Reads loss.json written by train.py after each epoch
# Format: [{"epoch": 1, "train_loss": ..., "val_loss": ...}, ...]

exp_root = Path("../experiments")

fig, ax = plt.subplots(figsize=(7, 4))
for variant in ["A", "B", "C", "D", "E"]:
    run_dirs = sorted(exp_root.glob(f"variant_{variant}_*"))
    if not run_dirs:
        continue
    log_path = run_dirs[0] / "loss.json"
    if not log_path.exists():
        continue
    log = json.loads(log_path.read_text())
    epochs   = [e["epoch"]    for e in log]
    val_loss = [e["val_loss"] for e in log]
    ax.plot(epochs, val_loss, label=f"Variant {variant}")

ax.set_xlabel("Epoch")
ax.set_ylabel("Validation CTC Loss")
ax.set_title("Validation loss across ablation variants")
ax.legend()
plt.tight_layout()
plt.savefig("../experiments/training_curves.pdf", bbox_inches="tight")
plt.show()